[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/27_deep_learning_foundations/01_perceptron_and_neuron.ipynb)

# Bài 01 — Perceptron và nơ-ron nhân tạo

**🎯 Mục tiêu**
- Nơ-ron nhân tạo: tổng có trọng số + hàm kích hoạt; quan hệ với hồi quy tuyến tính/logistic (chương 06–07).
- **Perceptron** (Rosenblatt, 1958): quy tắc học from scratch, định lý hội tụ, và thất bại với **XOR**.
- Vì sao cần **lớp ẩn**: giải XOR bằng cách ghép các nơ-ron; mạng nhiều lớp như phép biến đổi không gian.
- Trực giác **xấp xỉ vạn năng**: tổng của nhiều nơ-ron ReLU xấp xỉ được hàm liên tục.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/27_deep_learning_foundations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.datasets import make_classification_2d
from ml_course.visualization import plot_decision_boundary, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Nơ-ron nhân tạo

$$z = \mathbf w^\top\mathbf x + b, \qquad a = g(z)$$

| Hàm kích hoạt $g$ | Nơ-ron tương đương với |
|---|---|
| đồng nhất $g(z) = z$ | hồi quy tuyến tính (chương 06) |
| sigmoid $\sigma(z)$ | hồi quy logistic (chương 07) |
| bậc thang $\mathbb 1[z \ge 0]$ | **perceptron** |
| softmax (nhiều nơ-ron) | hồi quy softmax (AIVN tuần 5) |

Một nơ-ron đơn lẻ luôn tạo **ranh giới tuyến tính** $\mathbf w^\top\mathbf x + b = 0$. Sức mạnh của mạng nơ-ron đến từ việc **xếp chồng** nhiều nơ-ron với hàm kích hoạt **phi tuyến**.

## 2. Perceptron: quy tắc học

Nhãn $y \in \{-1, +1\}$, dự đoán $\hat y = \text{sign}(\mathbf w^\top\mathbf x + b)$. Duyệt từng điểm; khi **sai**:
$$\mathbf w \leftarrow \mathbf w + \eta\,y\,\mathbf x, \qquad b \leftarrow b + \eta\,y$$
Trực giác: điểm dương bị đoán âm → kéo $\mathbf w$ về phía $\mathbf x$ để $\mathbf w^\top\mathbf x$ tăng.

In [ ]:
class Perceptron:
    def __init__(self, lr=1.0, max_epochs=100, seed=0):
        self.lr, self.max_epochs, self.seed = lr, max_epochs, seed

    def fit(self, X, y):
        r = np.random.default_rng(self.seed)
        self.w_, self.b_ = np.zeros(X.shape[1]), 0.0
        self.mistakes_ = []
        for _ in range(self.max_epochs):
            errors = 0
            for i in r.permutation(len(X)):
                if y[i] * (X[i] @ self.w_ + self.b_) <= 0:              # sai (hoặc nằm đúng trên ranh giới)
                    self.w_ += self.lr * y[i] * X[i]
                    self.b_ += self.lr * y[i]
                    errors += 1
            self.mistakes_.append(errors)
            if errors == 0:                                             # đã tách hoàn hảo → dừng
                break
        return self

    def predict(self, X):
        return np.where(X @ self.w_ + self.b_ >= 0, 1, -1)


X_lin, y_lin = make_classification_2d("linear", n=200, noise=0.0, seed=1)
y_lin = np.where(y_lin == 1, 1, -1)
p = Perceptron().fit(X_lin, y_lin)
print(f"dữ liệu tách được tuyến tính: hội tụ sau {len(p.mistakes_)} epoch | số lỗi mỗi epoch: {p.mistakes_}")
print(f"accuracy train = {np.mean(p.predict(X_lin) == y_lin):.3f}")

**Định lý hội tụ (Novikoff, 1962):** nếu dữ liệu tách được với lề $\gamma$ (khoảng cách nhỏ nhất từ điểm đến một siêu phẳng tách) và mọi $\lVert\mathbf x\rVert \le R$, perceptron mắc **tối đa** $(R/\gamma)^2$ lỗi — bất kể thứ tự dữ liệu. Lề càng hẹp, càng nhiều bước. Nhưng nếu dữ liệu **không** tách được tuyến tính, perceptron **không bao giờ dừng**:

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
y_xor = np.array([-1, 1, 1, -1])                                        # XOR
p_xor = Perceptron(max_epochs=50).fit(X_xor, y_xor)
print(f"XOR: sau {len(p_xor.mistakes_)} epoch vẫn còn lỗi mỗi epoch: {p_xor.mistakes_[-5:]} | accuracy = {np.mean(p_xor.predict(X_xor) == y_xor):.2f}")

Không đường thẳng nào tách được XOR — Minsky & Papert (1969) chỉ ra giới hạn này, góp phần dẫn đến "mùa đông AI" đầu tiên.

## 3. Lớp ẩn giải XOR

XOR = (x₁ OR x₂) AND NOT (x₁ AND x₂). Mỗi phép logic là **một** nơ-ron bậc thang:

In [ ]:
step = lambda z: (z >= 0).astype(float)
W1 = np.array([[1, 1], [1, 1]], float)                                  # cột 0: OR, cột 1: AND
b1 = np.array([-0.5, -1.5])
w2, b2 = np.array([1.0, -1.0]), -0.5                                    # OR AND NOT(AND)
H = step(X_xor @ W1 + b1)
out = step(H @ w2 + b2)
for x, h, o in zip(X_xor, H, out):
    print(f"x = {x.astype(int)} → lớp ẩn (OR, AND) = {h.astype(int)} → XOR = {int(o)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
colors = np.where(y_xor > 0, "C1", "C0")
axes[0].scatter(*X_xor.T, c=colors, s=200)
axes[0].set_title("Không gian gốc: XOR không tách được")
axes[1].scatter(*(H + rng.normal(0, 0.02, H.shape)).T, c=colors, s=200)
axes[1].plot([0, 1], [-0.5, 0.5], "k--")
axes[1].set_xlabel("h₁ = OR")
axes[1].set_ylabel("h₂ = AND")
axes[1].set_title("Không gian ẩn: tách được bằng một đường thẳng")
plt.show()

Lớp ẩn **biến đổi** dữ liệu sang không gian mới, nơi bài toán trở nên tách được tuyến tính — đúng tinh thần kernel trick (chương 13), nhưng ở đây phép biến đổi được **học** từ dữ liệu (bài 02–03). Hàm bậc thang không có đạo hàm hữu ích → thực tế dùng sigmoid, tanh, ReLU (chương 28).

## 4. Trực giác xấp xỉ vạn năng

**Định lý xấp xỉ vạn năng** (Cybenko, 1989; Hornik, 1991): mạng **một** lớp ẩn với đủ nhiều nơ-ron (kích hoạt phi tuyến phù hợp) xấp xỉ được mọi hàm liên tục trên miền bị chặn với độ chính xác tuỳ ý.

Minh hoạ: dùng $N$ nơ-ron ReLU $\max(0, x - c_j)$ với các "điểm gãy" $c_j$ cố định, chỉ học trọng số lớp ra (bình phương tối thiểu) → hàm tuyến tính từng khúc xấp xỉ $f(x) = \sin(2x) + 0.3x$:

In [ ]:
f = lambda x: np.sin(2 * x) + 0.3 * x
xs = np.linspace(-3, 3, 400)
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
for ax, n_hidden in zip(axes, [3, 6, 12, 48]):
    knots = np.linspace(-3, 3, n_hidden, endpoint=False)
    H = np.maximum(0, xs[:, None] - knots[None])
    H = np.column_stack([np.ones_like(xs), H])                         # thêm bias của lớp ra
    w_out, *_ = np.linalg.lstsq(H, f(xs), rcond=None)
    approx = H @ w_out
    ax.plot(xs, f(xs), "k--", label="f(x)")
    ax.plot(xs, approx, "r", label=f"{n_hidden} nơ-ron ReLU")
    ax.set_title(f"{n_hidden} nơ-ron: sai số max = {np.max(np.abs(approx - f(xs))):.3f}")
    print(f"{n_hidden:2d} nơ-ron ReLU: sai số tuyệt đối lớn nhất = {np.max(np.abs(approx - f(xs))):.4f}")
    ax.legend(fontsize=8)
plt.show()

Càng nhiều nơ-ron, càng nhiều "khúc" → xấp xỉ càng tốt. Nhưng định lý chỉ nói **tồn tại** trọng số tốt; nó **không** nói:
- tìm được chúng bằng gradient descent,
- cần bao nhiêu nơ-ron (có thể là số mũ theo số chiều),
- mạng sẽ **tổng quát hoá** tốt ngoài dữ liệu train.

Thực tế, mạng **sâu** (nhiều lớp, mỗi lớp vừa phải) biểu diễn nhiều hàm hiệu quả hơn hẳn mạng nông rất rộng — các lớp sau tái sử dụng đặc trưng của lớp trước.

## 5. Mạng nhiều lớp như phép biến đổi dần

Với dữ liệu hai vòng tròn đồng tâm, một nơ-ron tuyến tính thất bại; một lớp ẩn nhỏ (dùng `MLPScratch` của thư viện — bài 03 tự cài đặt) đủ để "uốn" ranh giới:

In [ ]:
from ml_course.algorithms import MLPScratch
from sklearn.linear_model import LogisticRegression

X_c, y_c = make_classification_2d("circles", n=400, noise=0.1, seed=0)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
models = [("1 nơ-ron (logistic)", LogisticRegression()),
          ("1 lớp ẩn, 3 nơ-ron", MLPScratch(hidden=(3,), lr=0.05, epochs=300, seed=0)),
          ("1 lớp ẩn, 16 nơ-ron", MLPScratch(hidden=(16,), lr=0.02, epochs=300, seed=0))]
for ax, (name, m) in zip(axes, models):
    m.fit(X_c, y_c)
    plot_decision_boundary(m, X_c, y_c, ax=ax)
    ax.set_title(f"{name}: accuracy = {np.mean(m.predict(X_c) == y_c):.3f}")
    print(f"{name:22s} accuracy train = {np.mean(m.predict(X_c) == y_c):.3f}")
plt.show()

## 🧠 Kiểm tra nhanh

1. Một nơ-ron với sigmoid tương đương mô hình nào? Ranh giới quyết định của nó có dạng gì?
2. Viết quy tắc cập nhật perceptron. Khi nào nó chắc chắn hội tụ?
3. Vì sao perceptron đơn không học được XOR? Lớp ẩn giải quyết thế nào?
4. Định lý xấp xỉ vạn năng nói gì, và **không** nói gì?
5. Vì sao thực tế không dùng hàm bậc thang?

## 📌 Tóm tắt

- Nơ-ron = tổng có trọng số + kích hoạt; một nơ-ron cho ranh giới tuyến tính.
- Perceptron hội tụ khi dữ liệu tách được tuyến tính (≤ (R/γ)² lỗi), thất bại với XOR.
- Lớp ẩn biến đổi không gian; một lớp ẩn đủ rộng xấp xỉ mọi hàm liên tục — nhưng học được và tổng quát hoá là chuyện khác.

## 📚 Tài liệu tham khảo

- Rosenblatt (1958) — *The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain*
- Minsky & Papert (1969) — *Perceptrons*
- Goodfellow, Bengio & Courville (2016) — *Deep Learning*, ch. 6 (deeplearningbook.org)
- Nielsen — *Neural Networks and Deep Learning*, ch. 4 (chứng minh trực quan xấp xỉ vạn năng)